In [ ]:
# =========================================================
# NEW INFERENCE CELL: custom checkpoint path per model tag
# =========================================================
import glob
import os
import torch
import pandas as pd
import timm
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode

# ========================
# CONFIG
# ========================
test_dir = r"/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026/test_speedup"
checkpoint_dir = "fold_checkpoints"
output_csv = "my_submission_no_tta.csv"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Put custom model paths here for each tag.
# - Value can be a single string path or a list of paths.
# - If a tag is missing/empty, code falls back to checkpoint_dir pattern.
MODEL_PATHS = {
    "b0": [
        r"/kaggle/working/fold_checkpoints/rice_b0_single.pth",
        # r"/kaggle/working/rice_b0_fold0.pth",
    ],
    "b2": [
        r"/kaggle/working/fold_checkpoints/rice_b2_single.pth",
        # r"/kaggle/working/rice_b2_fold0.pth",
    ],
}

imagenet_mean = (0.485, 0.456, 0.406)
imagenet_std = (0.229, 0.224, 0.225)


# ========================
# TRANSFORM (NO TTA)
# ========================
def make_infer_transform(img_size):
    return transforms.Compose([
        transforms.Resize((img_size, img_size), interpolation=InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std),
    ])


# ========================
# LOAD MODELS
# ========================
def resolve_checkpoint_paths(tag, custom_paths_map=None):
    custom_paths_map = custom_paths_map or {}
    custom = custom_paths_map.get(tag)

    # Normalize custom input to a list and keep only existing files.
    if isinstance(custom, str):
        custom = [custom]
    elif custom is None:
        custom = []

    custom = [p for p in custom if p and os.path.exists(p)]
    if custom:
        return custom

    single_path = os.path.join(checkpoint_dir, f"rice_{tag}_single.pth")
    if os.path.exists(single_path):
        return [single_path]

    pattern = os.path.join(checkpoint_dir, f"rice_{tag}_fold*.pth")
    return sorted(glob.glob(pattern))


def load_model_group(tag, custom_paths_map=None):
    checkpoint_paths = resolve_checkpoint_paths(tag, custom_paths_map)

    if not checkpoint_paths:
        raise FileNotFoundError(
            f"No checkpoints found for tag={tag}. "
            f"Check MODEL_PATHS['{tag}'] or files in {checkpoint_dir}."
        )

    models = []
    class_names = None
    img_size = None
    fold_scores = []

    for path in checkpoint_paths:
        ckpt = torch.load(path, map_location=device)

        class_names = ckpt["class_names"]
        img_size = ckpt["img_size"]
        score = ckpt.get("split_score", ckpt.get("fold_score", 1.0))
        fold_scores.append(score)

        model = timm.create_model(
            ckpt["model_name"],
            pretrained=False,
            num_classes=len(class_names)
        )
        model.load_state_dict(ckpt["model_state_dict"], strict=True)
        model.to(device)
        model.eval()
        models.append(model)

    mean_score = sum(fold_scores) / len(fold_scores)

    print(f"[{tag.upper()}] Loaded {len(models)} model(s), score={mean_score:.4f}")
    for p in checkpoint_paths:
        print(f"  - {p}")

    return {
        "models": models,
        "class_names": class_names,
        "transform": make_infer_transform(img_size),
        "score": mean_score,
    }


# ========================
# SINGLE FORWARD (NO TTA)
# ========================
@torch.inference_mode()
def model_group_logits(group, pil_image):
    x = group["transform"](pil_image).unsqueeze(0).to(device)

    logits = []
    for model in group["models"]:
        logits.append(model(x))

    return torch.mean(torch.stack(logits, dim=0), dim=0)


# ========================
# LOAD
# ========================
group_b0 = load_model_group("b0", MODEL_PATHS)
group_b2 = load_model_group("b2", MODEL_PATHS)

if group_b0["class_names"] != group_b2["class_names"]:
    raise ValueError("Class mismatch")

# weight by validation score
score_sum = group_b0["score"] + group_b2["score"]
w_b0 = group_b0["score"] / score_sum
w_b2 = group_b2["score"] / score_sum

print(f"Weights: b0={w_b0:.4f}, b2={w_b2:.4f}")


# ========================
# PREDICT
# ========================
results = []

for img_name in sorted(os.listdir(test_dir)):
    if not img_name.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    path = os.path.join(test_dir, img_name)
    image = Image.open(path).convert("RGB")

    logits_b0 = model_group_logits(group_b0, image)
    logits_b2 = model_group_logits(group_b2, image)

    final_logits = w_b0 * logits_b0 + w_b2 * logits_b2
    pred = final_logits.argmax(dim=1).item()
    
    results.append([img_name, pred])


# ========================
# SAVE
# ========================
df = pd.DataFrame(results, columns=["image_id", "label"])
df.to_csv(output_csv, index=False)

print(f"Saved: {output_csv}")